# Improving Retrieval

## 1. Re-Ranking

In the context of RAG (Retrieval-Augmented Generation), reranking of retrieval results is a crucial step that refines the initial set of retrieved documents based on their relevance to the input query. This process involves re-scoring the retrieved documents using a more sophisticated model, such as a cross-encoder, to better capture the semantic similarity between the query and the documents. The reranked list of documents is then used as input for the generation model, ensuring that the most relevant and accurate information is utilized to generate the final output.

![Cross Encoder Image](https://raw.githubusercontent.com/UKPLab/sentence-transformers/master/docs/img/CrossEncoder.png)


Read more [here](https://www.sbert.net/examples/applications/retrieve_rerank/README.html)

Here are the steps:
* [Loading the reranking model](#loading-the-reranking-model)
* [Loading retrieval results](#loading-retrieval-results)
* [Calculating reranking score](#calculating-the-re-ranking-scores)
* [Generating a reply on the reranked documents](#using-merged-results-to-generate-a-reply)

In [19]:
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

### Loading the Reranking model

In [20]:
from sentence_transformers import CrossEncoder 
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print(cross_encoder.model)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 384, padding_idx=0)
      (position_embeddings): Embedding(512, 384)
      (token_type_embeddings): Embedding(2, 384)
      (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-5): 6 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=384, out_features=384, bias=True)
              (key): Linear(in_features=384, out_features=384, bias=True)
              (value): Linear(in_features=384, out_features=384, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=384, out_features=384, bias=True)
              (LayerNorm): LayerNorm((384,), eps=1e-1

### Loading retrieval results

We will load the retrieval results from the previous Hybrid-Search dataset.

In [21]:
import json
dataset = json.load(open('data/hybrid_search_results.json'))

In [4]:
dataset

{'15': {'id': 15,
  'text': "3.8 â Mixtral_8x7B 3.5 32 > $3.0 i] 228 fos a 2.0 0 5k 10k 15k 20k 25k 30k Context length Passkey Performance ry 3.8 â Mixtral_8x7B 3.5 0.8 32 > 0.6 $3.0 i] 228 04 fos 0.2 a 2.0 0.0 OK 4K 8K 12K 16K 20K 24K 28K 0 5k 10k 15k 20k 25k 30k Seq Len Context length Figure 4: Long range performance of Mixtral. (Left) Mixtral has 100% retrieval accuracy of the Passkey task regardless of the location of the passkey and length of the input sequence. (Right) The perplexity of Mixtral on the proof-pile dataset decreases monotonically as the context length increases.\n\nThe chunk discusses the long-range performance of the Mixtral model, demonstrating its ability to retrieve a passkey regardless of its location in a long input sequence, and showing that the model's perplexity on the proof-pile dataset decreases as the context length increases.",
  'metadata': {'title': 'Mixtral of Experts',
   'arxiv_id': '2401.04088',
   'references': ['1905.07830']}},
 '4': {'id': 4,
 

In [22]:
# This is the query that we used for the retrieval of the above documents
query = "What is context size of Mixtral?"

### Calculating the re-ranking scores

We are using the `cross_encoder` to calculate the match score.

In [23]:
pairs = [[query, doc['text']] for doc in dataset.values()] 
scores = cross_encoder.predict(pairs) 

print(scores)

[  5.065694    3.368832    7.1048393  -4.1161065  -4.375498   -5.261078
  -3.7225747   3.1854544   1.7966686  -2.5144243   2.5638714   2.3361564
  -3.0395935   3.08699   -11.0860405  -8.857571 ]


### Selecting top 3 reranked documents

In [24]:
# Combine scores with corresponding document IDs
results_with_scores = [
    (doc_id, dataset[doc_id]['text'], score)
    for doc_id, score in zip(dataset.keys(), scores)
]

# Sort results by score in descending order and take the top 3
top_results = sorted(results_with_scores, key=lambda x: x[2], reverse=True)[:3]

In [25]:
top_results

[('2',
  'expertsâ ) to process the token and combine their output additively. This technique increases the number of parameters of a model while controlling cost and latency, as the model only uses a fraction of the total set of parameters per token. Mixtral is pretrained with multilingual data using a context size of 32k tokens. It either matches or exceeds the performance of Llama 2 70B and GPT-3.5, over several benchmarks. In particular, Mixture of Experts Layer i gating inputs af outputs router expert\n\nThis chunk describes the key architectural details of the Mixtral model, a sparse mixture-of-experts language model that outperforms larger models like Llama 2 70B and GPT-3.5 on various benchmarks.',
  np.float32(7.1048393)),
 ('15',
  "3.8 â Mixtral_8x7B 3.5 32 > $3.0 i] 228 fos a 2.0 0 5k 10k 15k 20k 25k 30k Context length Passkey Performance ry 3.8 â Mixtral_8x7B 3.5 0.8 32 > 0.6 $3.0 i] 228 04 fos 0.2 a 2.0 0.0 OK 4K 8K 12K 16K 20K 24K 28K 0 5k 10k 15k 20k 25k 30k Seq Len Con

In [26]:
import numpy as np


# Add rows to the table with top 3 results
for doc_id, text, score in top_results:
    print("ID:", doc_id)
    print("Score:", score)
    print("Document:", text)
    print("--------------------------------")


ID: 2
Score: 7.1048393
Document: expertsâ ) to process the token and combine their output additively. This technique increases the number of parameters of a model while controlling cost and latency, as the model only uses a fraction of the total set of parameters per token. Mixtral is pretrained with multilingual data using a context size of 32k tokens. It either matches or exceeds the performance of Llama 2 70B and GPT-3.5, over several benchmarks. In particular, Mixture of Experts Layer i gating inputs af outputs router expert

This chunk describes the key architectural details of the Mixtral model, a sparse mixture-of-experts language model that outperforms larger models like Llama 2 70B and GPT-3.5 on various benchmarks.
--------------------------------
ID: 15
Score: 5.065694
Document: 3.8 â Mixtral_8x7B 3.5 32 > $3.0 i] 228 fos a 2.0 0 5k 10k 15k 20k 25k 30k Context length Passkey Performance ry 3.8 â Mixtral_8x7B 3.5 0.8 32 > 0.6 $3.0 i] 228 04 fos 0.2 a 2.0 0.0 OK 4K 8K 12K 16K 

### Using merged results to generate a reply

We can now take the improved merged results and call the LLM to generate the reply to the user's query.

In [27]:
# define a variable to hold the search results for the generation model
search_results = [doc[1] for doc in top_results]

In [29]:
search_results

['expertsâ ) to process the token and combine their output additively. This technique increases the number of parameters of a model while controlling cost and latency, as the model only uses a fraction of the total set of parameters per token. Mixtral is pretrained with multilingual data using a context size of 32k tokens. It either matches or exceeds the performance of Llama 2 70B and GPT-3.5, over several benchmarks. In particular, Mixture of Experts Layer i gating inputs af outputs router expert\n\nThis chunk describes the key architectural details of the Mixtral model, a sparse mixture-of-experts language model that outperforms larger models like Llama 2 70B and GPT-3.5 on various benchmarks.',
 "3.8 â Mixtral_8x7B 3.5 32 > $3.0 i] 228 fos a 2.0 0 5k 10k 15k 20k 25k 30k Context length Passkey Performance ry 3.8 â Mixtral_8x7B 3.5 0.8 32 > 0.6 $3.0 i] 228 04 fos 0.2 a 2.0 0.0 OK 4K 8K 12K 16K 20K 24K 28K 0 5k 10k 15k 20k 25k 30k Seq Len Context length Figure 4: Long range performanc

In [28]:
from dotenv import load_dotenv

load_dotenv()

True

In [30]:
# Now time to connect to the large language model
from openai import OpenAI

client = OpenAI()
model_name = "gpt-4.1-mini-2025-04-14"
completion = client.chat.completions.create(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are chatbot, an research expert. Your top priority is to help guide users to understand reserach papers."},
        {"role": "user", "content": query},
        {"role": "assistant", "content": str(search_results)}
    ]
)

response_text = str(completion.choices[0].message.content)

In [31]:
print(f"Hybrid Search with Reranking Reply to {query}: {response_text}")

Hybrid Search with Reranking Reply to What is context size of Mixtral?: The Mixtral model supports a fully dense context length of **32,000 tokens (32k tokens)**.


## 2. Re-packing

In the previous section, Re-Ranking, we successfully sorted our initial retrieved documents to find the Top 3 most relevant chunks. We then concatenated these full chunks and passed them to the LLM.

This is good, but it has two potential problems:

1. Context Window Limits: Our three chunks might be too large to fit into the LLM's context window. What if we had 10 relevant chunks?

2. Noise & "Lost in the Middle": LLMs can get "distracted" by irrelevant information within a document. Even in our top-ranked chunk (ID 2), only one sentence is directly relevant to the query. The rest is noise.

Re-packing (or "compression") solves this. It's the process of taking the top-ranked documents and extracting only the most relevant snippets from them before sending them to the final generation LLM.

Re-Ranking vs. Re-Packing:

* Re-Ranking: Sorts a list of whole documents. (e.g., "Document 2 is better than Document 5.")

* Re-Packing: Pulls specific sentences out of those documents. (e.g., "Sentence 3 from Document 2 is the only part I need.")

We will use an LLM as an "extractor" to perform this re-packing.

In [32]:
# Define the extraction prompt for re-packing
EXTRACTION_PROMPT = """You are a precise text extraction assistant. Your task is to extract ONLY the sentences or phrases from the given document that are directly relevant to answering the user's query.

Rules:
1. Extract exact sentences/phrases from the document - do not paraphrase or summarize
2. Only extract content that directly helps answer the query
3. If multiple relevant sentences exist, include all of them
4. If no content in the document is relevant to the query, respond with: "NO_RELEVANT_CONTENT"
5. Preserve the original wording exactly as it appears in the document

Query: {query}

Document:
{document}

Relevant extractions:"""


In [33]:
def extract_relevant_snippets(query: str, document: str, client: OpenAI, model: str = "gpt-4.1-mini-2025-04-14") -> str:
    """
    Extract only the relevant snippets from a document given a query.
    
    Args:
        query: The user's search query
        document: The full document text to extract from
        client: OpenAI client instance
        model: The model to use for extraction
        
    Returns:
        Extracted relevant snippets or "NO_RELEVANT_CONTENT" if nothing relevant
    """
    prompt = EXTRACTION_PROMPT.format(query=query, document=document)
    
    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0  # Use deterministic output for extraction
    )
    
    return completion.choices[0].message.content.strip()


### Applying extraction to top-ranked documents

Now we'll apply our extraction function to each of the top 3 documents from the re-ranking step. This will pull out only the sentences that are directly relevant to our query.


In [34]:
# Apply extraction to each top-ranked document
extracted_snippets = []

for doc_id, text, score in top_results:
    print(f"Extracting from Document ID: {doc_id} (score: {score:.2f})")
    print("-" * 50)
    
    snippet = extract_relevant_snippets(query, text, client, model_name)
    extracted_snippets.append({
        "doc_id": doc_id,
        "original_text": text,
        "extracted_snippet": snippet,
        "score": score
    })
    
    print(f"Extracted: {snippet}")
    print("=" * 70)
    print()


Extracting from Document ID: 2 (score: 7.10)
--------------------------------------------------
Extracted: Mixtral is pretrained with multilingual data using a context size of 32k tokens.

Extracting from Document ID: 15 (score: 5.07)
--------------------------------------------------
Extracted: 3.8 â Mixtral_8x7B 3.5 32 > $3.0 i] 228 fos a 2.0 0 5k 10k 15k 20k 25k 30k Context length  
3.8 â Mixtral_8x7B 3.5 0.8 32 > 0.6 $3.0 i] 228 04 fos 0.2 a 2.0 0.0 OK 4K 8K 12K 16K 20K 24K 28K  
Figure 4: Long range performance of Mixtral. (Left) Mixtral has 100% retrieval accuracy of the Passkey task regardless of the location of the passkey and length of the input sequence. (Right) The perplexity of Mixtral on the proof-pile dataset decreases monotonically as the context length increases.

Extracting from Document ID: 4 (score: 3.37)
--------------------------------------------------
Extracted: "Mixtral supports a fully dense context length of 32k tokens"



### Comparing context sizes: Before vs After Re-packing

Let's see how much we've reduced the context size by extracting only the relevant snippets.


In [35]:
# Calculate original context size (all top documents concatenated)
original_context = "\n\n".join([doc[1] for doc in top_results])
original_char_count = len(original_context)

# Calculate re-packed context size (only extracted snippets)
# Filter out "NO_RELEVANT_CONTENT" responses
relevant_snippets = [
    s["extracted_snippet"] 
    for s in extracted_snippets 
    if s["extracted_snippet"] != "NO_RELEVANT_CONTENT"
]
repacked_context = "\n\n".join(relevant_snippets)
repacked_char_count = len(repacked_context)

# Calculate reduction percentage
reduction_pct = ((original_char_count - repacked_char_count) / original_char_count) * 100

print("=" * 60)
print("CONTEXT SIZE COMPARISON")
print("=" * 60)
print(f"Original context size:   {original_char_count:,} characters")
print(f"Re-packed context size:  {repacked_char_count:,} characters")
print(f"Reduction:               {reduction_pct:.1f}%")
print("=" * 60)
print(f"\nDocuments with relevant content: {len(relevant_snippets)}/{len(extracted_snippets)}")


CONTEXT SIZE COMPARISON
Original context size:   2,593 characters
Re-packed context size:  637 characters
Reduction:               75.4%

Documents with relevant content: 3/3


### Generating a response using re-packed context

Now we'll use only the extracted relevant snippets to generate our final response. This should give us a more focused answer with less noise.


In [36]:
# Generate response using the re-packed (extracted) context
repacked_completion = client.chat.completions.create(
    model=model_name,
    messages=[
        {
            "role": "system", 
            "content": "You are a research expert. Answer the user's question based ONLY on the provided context. Be concise and accurate."
        },
        {
            "role": "user", 
            "content": f"Context:\n{repacked_context}\n\nQuestion: {query}"
        }
    ]
)

repacked_response = repacked_completion.choices[0].message.content
print(f"Re-packed Response to '{query}':\n")
print(repacked_response)


Re-packed Response to 'What is context size of Mixtral?':

The context size of Mixtral is 32,000 tokens.


### Comparing Results: Re-Ranking vs Re-Packing

Let's compare the responses generated using:
1. **Re-Ranking only**: Full documents, sorted by relevance
2. **Re-Ranking + Re-Packing**: Only extracted relevant snippets


In [37]:
print("=" * 70)
print("COMPARISON: Re-Ranking Only vs Re-Ranking + Re-Packing")
print("=" * 70)

print("\n" + "-" * 70)
print("METHOD 1: Re-Ranking Only (Full Documents)")
print("-" * 70)
print(f"Context size: {original_char_count:,} characters")
print(f"Response: {response_text}")

print("\n" + "-" * 70)
print("METHOD 2: Re-Ranking + Re-Packing (Extracted Snippets)")
print("-" * 70)
print(f"Context size: {repacked_char_count:,} characters ({reduction_pct:.1f}% smaller)")
print(f"Response: {repacked_response}")

print("\n" + "=" * 70)
print("KEY TAKEAWAYS")
print("=" * 70)
print(f"1. Context reduced by {reduction_pct:.1f}% while preserving relevant information")
print("2. Re-packing removes noise that could distract the LLM")
print("3. Smaller context = faster inference and lower costs")
print("4. Both methods should produce similar quality answers for this query")


COMPARISON: Re-Ranking Only vs Re-Ranking + Re-Packing

----------------------------------------------------------------------
METHOD 1: Re-Ranking Only (Full Documents)
----------------------------------------------------------------------
Context size: 2,593 characters
Response: The Mixtral model supports a fully dense context length of **32,000 tokens (32k tokens)**.

----------------------------------------------------------------------
METHOD 2: Re-Ranking + Re-Packing (Extracted Snippets)
----------------------------------------------------------------------
Context size: 637 characters (75.4% smaller)
Response: The context size of Mixtral is 32,000 tokens.

KEY TAKEAWAYS
1. Context reduced by 75.4% while preserving relevant information
2. Re-packing removes noise that could distract the LLM
3. Smaller context = faster inference and lower costs
4. Both methods should produce similar quality answers for this query


### Summary: Re-Packing Benefits

**When to use Re-Packing:**
- When you have many retrieved documents that exceed the LLM's context window
- When documents contain lots of irrelevant information (noise)
- When you want to reduce inference costs and latency
- When you're experiencing "lost in the middle" issues with long contexts

**Trade-offs:**
- Additional LLM call required for extraction (adds latency and cost)
- Risk of accidentally removing relevant information if extraction is imperfect
- Best used after initial retrieval and re-ranking have been done

**Best Practices:**
1. Use a smaller/cheaper model for extraction (e.g., gpt-4.1-mini)
2. Use temperature=0 for deterministic extraction
3. Always validate that extracted snippets actually answer the query
4. Consider batching extraction calls for efficiency
